# DP2 tract 9813: metadetect shear → Kaiser–Squires mass map

Contact author: Alex Broughton  
<br>Date: 


In [ ]:
! eups list -s | grep lsst_distrib

## Introduction

This notebook builds a **single-tract** weak-lensing convergence map (**κ**) for **tract 9813** from DP2 **metadetect** `object_shear_patch` catalogs via the Rubin Butler, using a **FFT Kaiser–Squires** inversion on a tangent plane centered on the tract.

**References**

- Butler access and dataset overview: [`_tutorials/access-dp2-metadetect-catalogs.ipynb`](../_tutorials/access-dp2-metadetect-catalogs.ipynb)
- Selection ideas (re-expressed here for DP2 columns): [lsst-so/comcam_clusters — `A360_shear_profile_metadetect.ipynb`](https://github.com/lsst-so/comcam_clusters/blob/main/A360_shear_profile_metadetect.ipynb), [`ACO360_SourceSelection.ipynb`](https://github.com/lsst-so/comcam_clusters/blob/main/ACO360_SourceSelection.ipynb)
- Mass mapping on small cutouts in that repo uses aperture statistics; this notebook uses **KS FFT** for tract-scale maps instead of the `ACO360_WL_HSCcalib_MassMap.ipynb` double loop. **Do not** use the ComCam **HSC shear calibration** path: DP2 metadetect uses the **noshear** branch (schema column **`ns`**, or `shear_type` / `mdet_step` when present).

**Caveats**

- Tangent-plane **flat-sky** approximation; one tract only.
- **Memory:** a full tract can be large—use `CONFIG["MAX_PATCHES"]` for quick tests and/or reduce `GRID_NX` × `GRID_NY`.
- Column names are **inferred** from the Arrow schema on first load; verify on SDF if inference fails.

**Future (not implemented here):** full-DP2 runs as parallel tract jobs, peak finding on saved κ arrays, and **N** shear-angle nulls with `numpy.random.Generator` seeded from `CONFIG["RANDOM_SEED"]` + job id—see markdown section 8.0.


## 1.0 Set up

`CONFIG` holds everything a future cluster script would mirror with CLI args. Core logic lives in importable pure functions in [`lib/tract_massmap.py`](lib/tract_massmap.py).


In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pyarrow as pa

from lsst.daf.butler import Butler
from lsst.skymap import BaseSkyMap

# Repo root: cwd is repo or notebooks/
for base in (Path.cwd(), Path.cwd().parent):
    if (base / "notebooks" / "lib" / "tract_massmap.py").is_file():
        if str(base / "notebooks") not in sys.path:
            sys.path.insert(0, str(base / "notebooks"))
        REPO_ROOT = base
        break
else:
    raise RuntimeError("Run from the git repo root or from notebooks/.")

from lib.tract_massmap import (
    ShearColumnMap,
    arrow_to_numpy_shear,
    bin_shear_weighted,
    columns_for_loading,
    filter_noshear_table,
    infer_shear_columns,
    kaiser_squires,
    list_patches_for_tract,
    load_patches_concat,
    select_sources_basic,
)

CONFIG = {
    "REPO": "/sdf/data/rubin/repo/dp2_prep",
    "COLLECTION": "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "SKYMAP": "lsst_cells_v2",
    "INSTRUMENT": "LSSTCam",
    "TRACT": 9813,
    # Grid / projection
    "GRID_NX": 512,
    "GRID_NY": 512,
    "PIXEL_SCALE_ARCSEC": 0.2,
    # Selection (tune for science)
    "MIN_WEIGHT": 0.0,
    "MAX_ABS_G": 1.0,
    # Debug: set to an int to load only the first N patches with data
    "MAX_PATCHES": None,
    # Reproducibility placeholder for future angle-null jobs (not used in v1)
    "RANDOM_SEED": 42,
}

%matplotlib inline
plt.rcParams.update({"font.size": 12})

print("REPO_ROOT =", REPO_ROOT)
print("TRACT =", CONFIG["TRACT"])


## 2.0 Butler and patch list

Query the registry for every `object_shear_patch` dataset in this tract (do not load `object_shear_all` whole-survey tables into memory).


In [ ]:
butler = Butler(
    CONFIG["REPO"],
    collections=[CONFIG["COLLECTION"]],
    skymap=CONFIG["SKYMAP"],
    instrument=CONFIG["INSTRUMENT"],
)

patches = list_patches_for_tract(butler, CONFIG["TRACT"], CONFIG["COLLECTION"])
print(f"Found {len(patches)} patches with object_shear_patch for tract {CONFIG['TRACT']}")
print("first few:", patches[:12])


## 3.0 Load catalogs (Arrow)

Load one patch to infer column names, then reload a minimal column subset for all patches (faster I/O, lower memory).


In [ ]:
# First patch with a row count (skip missing)
probe_patch = None
probe_tbl = None
for p in patches:
    if not butler.exists("object_shear_patch", dataId={"tract": CONFIG["TRACT"], "patch": p}):
        continue
    t = butler.get("object_shear_patch", dataId={"tract": CONFIG["TRACT"], "patch": p})
    if t.num_rows > 0:
        probe_patch, probe_tbl = p, t
        break

if probe_tbl is None:
    raise RuntimeError("No non-empty object_shear_patch in this tract (check collection / tract).")

print("Probe patch", probe_patch, "rows", probe_tbl.num_rows)
cmap = infer_shear_columns(probe_tbl.schema)
print("Inferred columns:", cmap)

cols = columns_for_loading(cmap)
raw = load_patches_concat(
    butler,
    CONFIG["TRACT"],
    patches,
    column_subset=cols,
    max_patches=CONFIG["MAX_PATCHES"],
)
print("Concatenated rows (all shear branches):", raw.num_rows)


## 4.0 Noshear branch (`ns`) and source selection

Keep **noshear** metadetect rows only, then apply conservative cuts in `select_sources_basic` (see `tract_massmap.py`).


In [ ]:
noshear_tbl = filter_noshear_table(raw, cmap)
print("Rows after noshear filter:", noshear_tbl.num_rows)

ra, dec, g1, g2, w = arrow_to_numpy_shear(noshear_tbl, cmap)
ra, dec, g1, g2, w = select_sources_basic(
    ra, dec, g1, g2, w,
    min_weight=CONFIG["MIN_WEIGHT"],
    max_abs_g=CONFIG["MAX_ABS_G"],
)
print("Sources after quality cuts:", ra.size)


## 5.0 Tangent plane and weighted binning

Tract center from the skymap (`ctr_coord`) defines the TAN projection origin. Pixels use `PIXEL_SCALE_ARCSEC`.


In [ ]:
sky_map: BaseSkyMap = butler.get("skyMap")
tract_info = sky_map[CONFIG["TRACT"]]
ctr = tract_info.ctr_coord
ra0 = ctr.getRa().asDegrees()
dec0 = ctr.getDec().asDegrees()
print(f"Tangent origin (tract center): ra={ra0:.6f} deg, dec={dec0:.6f} deg")

g1_map, g2_map, w_map, hits, wcs = bin_shear_weighted(
    ra,
    dec,
    g1,
    g2,
    w,
    ra0_deg=ra0,
    dec0_deg=dec0,
    pixel_scale_arcsec=CONFIG["PIXEL_SCALE_ARCSEC"],
    nx=CONFIG["GRID_NX"],
    ny=CONFIG["GRID_NY"],
)
print("Non-empty cells:", int((hits > 0).sum()), "/", hits.size)


## 6.0 Kaiser–Squires inversion

Pixel scale in **radians** enters the FFT wavevector grid.


In [ ]:
pixel_rad = np.deg2rad(CONFIG["PIXEL_SCALE_ARCSEC"] / 3600.0)
kappa_e, kappa_b = kaiser_squires(g1_map, g2_map, pixel_scale_rad=pixel_rad)
print("kappa_E range:", float(np.nanmin(kappa_e)), float(np.nanmax(kappa_e)))
print("kappa_B std (noise-like if small systematics):", float(np.std(kappa_b)))


## 7.0 Figures

E-mode κ (lensing signal) and B-mode map (approximate null).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

extent = [
    0.5,
    CONFIG["GRID_NX"] + 0.5,
    0.5,
    CONFIG["GRID_NY"] + 0.5,
]

kw = dict(origin="lower", interpolation="nearest", cmap="magma", extent=extent)
axes[0].imshow(np.log10(hits + 0.1), **kw)
axes[0].set_title("log10(hit count + 0.1)")
axes[1].imshow(kappa_e, **dict(kw, cmap="RdBu_r"))
axes[1].set_title(r"$\kappa_E$ (Kaiser–Squires)")
axes[2].imshow(kappa_b, **dict(kw, cmap="RdBu_r"))
axes[2].set_title(r"$\kappa_B$ (null)")

for ax in axes:
    ax.set_xlabel("pixel x")
    ax.set_ylabel("pixel y")
fig.suptitle(f"Tract {CONFIG['TRACT']} — DP2 metadetect noshear")
fig.tight_layout()

fig2, ax = plt.subplots(figsize=(6, 4))
ax.hist(kappa_b.ravel(), bins=80, color="steelblue", alpha=0.85)
ax.set_xlabel(r"$\kappa_B$")
ax.set_ylabel("pixels")
ax.set_title(r"$\kappa_B$ pixel histogram (FFT null channel)")
fig2.tight_layout()


## 8.0 Future: scale, peaks, nulls, scripts

**Full DP2 footprint:** run **one job per tract** (or per tile), write `npz` / FITS / Zarr with κ, γ grids, and WCS for downstream stacking and correlation with survey-property maps.

**Peaks:** treat κ and optional κ_B as numpy arrays + `astropy.wcs.WCS` so a separate peak-finder step can consume saved maps.

**Shear-angle nulls:** for realization `i`, rotate `g1 + i g2` by a fixed or random angle with `numpy.random.Generator(CONFIG["RANDOM_SEED"] + i)`, re-bin, re-run KS; submit **N** independent cluster jobs.

**Notebook → script:** mirror `CONFIG` with `argparse` or YAML, `import lib.tract_massmap`, log Butler **collection**, **skymap**, tract list, cut thresholds, grid shape, and git hash / `eups` versions.
